## Custom convolution model in PyTorch

In [ ]:
!pip install openimages

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 33.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.2/82.2 kB 7.6 MB/s eta 0:00:00


In [ ]:
data_folder = "data"
sample_count = 3000
class_to_index = {"Car": 0, "House": 1, "Tower": 2}

In [ ]:
from openimages.download import download_dataset
import os

def download_openimages(data_folder, sample_count, classes):
  if not os.path.exists(data_folder):
      os.makedirs(data_folder)
  # Download equal amount of images per class if possible
  sample_count_per_class = sample_count // len(classes)
  download_dataset(data_folder, classes, limit=sample_count_per_class)

download_openimages(data_folder, sample_count, list(class_to_index.keys()))

100%|██████████| 1000/1000 [00:23<00:00, 42.28it/s]


In [ ]:
import torch
from torchvision import transforms, models, datasets

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor()
])

In [ ]:
class ConvNetwork(torch.nn.Module):
  def __init__(self, in_shape, out_class_count):
    super().__init__()
    self.conv1 = torch.nn.Conv2d(in_shape[0], 32, (3, 3), padding='same')
    self.bn1 = torch.nn.BatchNorm2d(32)
    self.pool1 = torch.nn.MaxPool2d((2, 2), (2, 2))
    self.conv2 = torch.nn.Conv2d(32, 64, (3, 3), padding='same')
    self.bn2 = torch.nn.BatchNorm2d(64)
    self.pool2 = torch.nn.MaxPool2d((2, 2), (2, 2))
    self.conv3 = torch.nn.Conv2d(64, 128, (3, 3), padding='same')
    self.bn3 = torch.nn.BatchNorm2d(128)
    self.pool3 = torch.nn.AvgPool2d((2, 2), (2, 2))
    self.fc4 = torch.nn.Linear(128 * (in_shape[1] // 8) * (in_shape[2] // 8), 256)
    self.bn4 = torch.nn.BatchNorm1d(256)
    self.fc5 = torch.nn.Linear(256, out_class_count)

  def forward(self, x):
    y = torch.nn.Sequential(
      self.conv1,
      self.bn1,
      torch.nn.ReLU(),
      self.pool1,
      self.conv2,
      self.bn2,
      torch.nn.ReLU(),
      self.pool2,
      self.conv3,
      self.bn3,
      torch.nn.ReLU(),
      self.pool3,
      torch.nn.Flatten(),
      self.fc4,
      self.bn4,
      torch.nn.ReLU(),
      self.fc5
    )(x)
    return y


model = ConvNetwork(in_shape=(3, 224, 224), out_class_count=len(class_to_index))
model = model.to(device)

In [ ]:
import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from random import shuffle

class ImageDataset(Dataset):
  def __init__(self, paths_with_class, transform):
    self.paths_with_class = paths_with_class
    self.transform = transform

  @classmethod
  def from_folder(cls, data_folder, class_to_index, transform=transforms.ToTensor()):
    paths_with_class = cls.paths_and_class_from_folder(data_folder, class_to_index)
    return cls(paths_with_class, transform)

  @classmethod
  def paths_and_class_from_folder(cls, data_folder, class_to_index):
    paths_with_class_index = []
    for class_name in class_to_index.keys():
      image_paths = glob.glob("{}/{}/images/*.jpg".format(data_folder, class_name.lower()))
      # make (path_to_image, true_class_index) pairs for shuffling and unequal image count per class
      paths_with_class_index += map(lambda paths: (paths, class_to_index[class_name]), list(image_paths))
    return paths_with_class_index

  def __len__(self):
    return len(self.paths_with_class)

  def __getitem__(self, index):
    path, image_class_index = self.paths_with_class[index]
    compatible_image = self.get_compatible_image(path)
    return (compatible_image, image_class_index)

  def split(self, train_transform, test_transform, train_percent=0.7, validate_percent=0.1):
    train_dataset_size = int(train_percent * len(self))
    validate_dataset_size = int(validate_percent * len(self))
    paths_with_class_copy = self.paths_with_class.copy()
    # split on shuffled data
    shuffle(paths_with_class_copy)
    train_dataset = ImageDataset(paths_with_class_copy[:train_dataset_size], train_transform)
    validate_dataset = ImageDataset(paths_with_class_copy[train_dataset_size:train_dataset_size+validate_dataset_size], test_transform)
    test_dataset = ImageDataset(paths_with_class_copy[train_dataset_size+validate_dataset_size:], test_transform)
    return (train_dataset, test_dataset, validate_dataset)

  def get_compatible_image(self, full_path):
    # have only 3 channels per pixel
    image = Image.open(full_path).convert('RGB')
    transformed_image = self.transform(image)
    return transformed_image

train_dataset, test_dataset, validate_dataset = ImageDataset \
  .from_folder(data_folder, class_to_index) \
  .split(train_transform, test_transform)

In [ ]:
import numpy as np
from datetime import datetime

In [ ]:
import numpy as np
from datetime import datetime

def train(model, dataloader, class_count, epoch_count=20, learning_rate=1e-4):
  loss_function = torch.nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
  model.train()
  start_time = datetime.now()
  for epoch in range(epoch_count):
    loss_acumulator = np.empty(len(dataloader), dtype=np.float32)
    i = 0
    for inputs, labels in dataloader:
      inputs = inputs.to(device)
      labels = torch.nn.functional.one_hot(labels, num_classes=class_count).float().to(device)
      next_i = i + dataloader.batch_size
      predictions = model(inputs)
      loss = loss_function(predictions, labels)
      loss_acumulator[i : next_i] = loss.cpu().detach().numpy()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
      i = next_i
    current_time = datetime.now()
    print(f'Epoch: {epoch}, Time: {current_time - start_time}, Loss: {np.mean(loss_acumulator)}')
    start_time = current_time

train_dataloader = DataLoader(train_dataset, batch_size=50, shuffle=True, num_workers=4)
train(model, train_dataloader, len(class_to_index))

Epoch: 0, Time: 0:00:37.835190, Loss: 1.0767453908920288
Epoch: 1, Time: 0:00:37.880887, Loss: 0.6533816456794739
Epoch: 2, Time: 0:00:36.283253, Loss: 0.7370682954788208
Epoch: 3, Time: 0:00:37.613508, Loss: 0.6621020436286926
Epoch: 4, Time: 0:00:37.820534, Loss: 0.7123875617980957
Epoch: 5, Time: 0:00:37.529340, Loss: 0.663536548614502
Epoch: 6, Time: 0:00:37.933035, Loss: 0.43541908264160156
Epoch: 7, Time: 0:00:37.780312, Loss: 0.44833219051361084
Epoch: 8, Time: 0:00:37.741724, Loss: 0.3284304440021515
Epoch: 9, Time: 0:00:37.564925, Loss: 0.524664580821991
Epoch: 10, Time: 0:00:36.655664, Loss: 0.3686792254447937
Epoch: 11, Time: 0:00:36.980290, Loss: 0.4412030279636383
Epoch: 12, Time: 0:00:38.077814, Loss: 0.5782127380371094
Epoch: 13, Time: 0:00:37.970250, Loss: 0.46890872716903687
Epoch: 14, Time: 0:00:37.399962, Loss: 0.35576289892196655
Epoch: 15, Time: 0:00:37.345266, Loss: 0.3636800944805145
Epoch: 16, Time: 0:00:37.385333, Loss: 0.3627449572086334
Epoch: 17, Time: 0:00:

In [ ]:
def test(model, dataloader, class_count):
  model.eval()
  dataset_size = len(dataloader) * dataloader.batch_size
  guesses_per_class = np.empty((dataset_size, class_count), dtype=np.float32)
  labels = np.empty(dataset_size, dtype=np.int16)
  i = 0
  start_time = datetime.now()
  for inputs, batch_labels in dataloader:
    inputs = inputs.to(device)
    with torch.no_grad():
      outputs = model(inputs)
    batch_probabilities = torch.sigmoid(outputs)
    # size(0) is current batch size
    next_i = i + inputs.size(0)
    guesses_per_class[i : next_i, : ] = batch_probabilities.detach().cpu().numpy()
    labels[i : next_i] = batch_labels
    i = next_i
  print(f"Time: {datetime.now() - start_time}")
  return (guesses_per_class, labels)

test_dataloader = DataLoader(test_dataset, batch_size=50, shuffle=False, num_workers=4)
test_results = test(model, test_dataloader, len(class_to_index))
test_results

Time: 0:00:07.871705


(array([[0.84548885, 0.3032419 , 0.07676161],
        [0.66123015, 0.6939526 , 0.06186252],
        [0.56021816, 0.63350475, 0.13843718],
        ...,
        [0.34104168, 0.5311985 , 0.6443716 ],
        [0.9068142 , 0.44421718, 0.06635845],
        [0.3294318 , 0.21096912, 0.9632227 ]], dtype=float32),
 array([0, 1, 0, 2, 2, 1, 1, 1, 0, 2, 2, 2, 1, 2, 2, 0, 1, 2, 2, 0, 0, 0,
        2, 0, 0, 1, 2, 2, 2, 0, 1, 1, 0, 2, 0, 2, 0, 2, 1, 2, 2, 1, 2, 1,
        1, 2, 1, 0, 1, 0, 0, 1, 2, 2, 2, 0, 1, 2, 0, 1, 0, 0, 1, 2, 0, 1,
        0, 1, 1, 0, 1, 0, 1, 2, 0, 0, 1, 0, 1, 0, 0, 1, 2, 2, 2, 0, 0, 1,
        1, 2, 1, 0, 0, 0, 1, 0, 0, 2, 1, 1, 2, 0, 2, 1, 1, 0, 1, 2, 1, 1,
        1, 2, 1, 0, 1, 1, 0, 2, 2, 0, 1, 2, 0, 0, 0, 1, 0, 1, 2, 0, 1, 1,
        2, 2, 0, 2, 0, 0, 1, 0, 0, 2, 1, 0, 1, 2, 2, 1, 0, 1, 1, 1, 2, 0,
        2, 2, 2, 2, 1, 0, 2, 1, 0, 0, 2, 0, 0, 0, 2, 0, 2, 2, 1, 1, 2, 2,
        1, 2, 1, 1, 1, 2, 2, 0, 0, 2, 0, 0, 0, 2, 0, 0, 2, 1, 2, 1, 1, 2,
        0, 1, 2, 1, 2, 0, 1,

In [ ]:
import math

def get_metrics(results, for_class_index, threshold=0.5):
  guesses_per_class, labels = results
  for_class_guesses = guesses_per_class[:, for_class_index]
  # Cached masks
  positive_guess = for_class_guesses > threshold
  true_for_class_index = labels == for_class_index
  tp = (
      positive_guess
      & true_for_class_index
      ).sum()
  fp = (
      positive_guess
      & ~true_for_class_index
      ).sum()
  fn = (
      ~positive_guess
      & true_for_class_index
      ).sum()
  data_size = len(labels)
  print(f"size={data_size}\ntp={tp}\nfp={fp}\nfn={fn}\ntn={data_size - tp - fp - fn}")
  accuracy = (data_size - fp - fn) / data_size
  precision = tp / (tp + fp)
  recall = tp / (tp + fn)
  f2 = (2 * precision * recall) / (precision + recall)
  return (accuracy, precision, recall, f2)

def print_metrics(results, for_class, class_to_index, threshold=0.5):
  class_index = class_to_index[for_class]
  accuracy, precision, recall, f2 = get_metrics(results, class_index, threshold)
  print(f"{for_class}, index = {class_index}, threshold = {threshold}:\n\
        accuracy: {accuracy:.2f}\n\
        precision: {precision:.2f}\n\
        recall: {recall:.2f}\n\
        f2: {f2:.2f}"
  )

# Classes: {"Car": 0, "House": 1, "Tower": 2}
thresholds = (0.2, 0.5, 0.8)
for_classes = ["Car"] * len(thresholds)
for for_class, threshold in zip(for_classes, thresholds):
  print_metrics(test_results, for_class, class_to_index, threshold)

size=600
tp=202
fp=217
fn=3
tn=178
Car, index = 0, threshold = 0.2:
        accuracy: 0.63
        precision: 0.48
        recall: 0.99
        f2: 0.65
size=600
tp=183
fp=49
fn=22
tn=346
Car, index = 0, threshold = 0.5:
        accuracy: 0.88
        precision: 0.79
        recall: 0.89
        f2: 0.84
size=600
tp=124
fp=2
fn=81
tn=393
Car, index = 0, threshold = 0.8:
        accuracy: 0.86
        precision: 0.98
        recall: 0.60
        f2: 0.75


In [ ]:
def guess(input):
  model.eval()
  input = input.to(device)
  with torch.no_grad():
    output = model(input)
  return output

In [ ]:
# save model
torch.save(model.state_dict(), "model.pth")

In [ ]:
!gdown --id 1V_uMcSu1MN0qjEkkqnwN9qEVOU27VyNz

/usr/local/lib/python3.10/dist-packages/gdown/cli.py:138: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1V_uMcSu1MN0qjEkkqnwN9qEVOU27VyNz
From (redirected): https://drive.google.com/uc?id=1V_uMcSu1MN0qjEkkqnwN9qEVOU27VyNz&confirm=t&uuid=8a992c84-d6fe-44a0-a4e5-cddad2a4c6b6
To: /content/model.pth
100% 103M/103M [00:00<00:00, 110MB/s]


In [ ]:
# load model
model.load_state_dict(torch.load("model.pth"))

<All keys matched successfully>